# 07 — Model Comparison
Compare Logistic Regression, Random Forest, XGBoost. Pick winner favoring recall. Save pipeline.

In [1]:

import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              roc_auc_score, classification_report)
import joblib

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed — will skip XGB")

PROC = r'../data/processed'
MODELS = r'../models'
os.makedirs(MODELS, exist_ok=True)

df = pd.read_csv(f'{PROC}/feature_matrix.csv')
target = 'AttritionRisk_Label'
drop_cols = [c for c in [target, 'EmployeeID'] if c in df.columns]
X = df.drop(columns=drop_cols).astype(float)
y = df[target]
print(f"X: {X.shape}, class balance: {y.value_counts().to_dict()}")


X: (500, 43), class balance: {0: 445, 1: 55}


In [2]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

def evaluate(name, pipeline, X_tr, y_tr, X_te, y_te):
    pipeline.fit(X_tr, y_tr)
    y_pred = pipeline.predict(X_te)
    y_prob = pipeline.predict_proba(X_te)[:, 1]
    return {
        'model': name,
        'precision': round(precision_score(y_te, y_pred), 4),
        'recall': round(recall_score(y_te, y_pred), 4),
        'f1': round(f1_score(y_te, y_pred), 4),
        'roc_auc': round(roc_auc_score(y_te, y_prob), 4),
        'pipeline': pipeline
    }

results = []

# ── Logistic Regression ──
lr = Pipeline([('sc', StandardScaler()),
               ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))])
results.append(evaluate('Logistic Regression', lr, X_train, y_train, X_test, y_test))

# ── Random Forest ──
rf = Pipeline([('sc', StandardScaler()),
               ('clf', RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                              random_state=42, n_jobs=-1))])
results.append(evaluate('Random Forest', rf, X_train, y_train, X_test, y_test))

# ── XGBoost ──
if HAS_XGB:
    scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
    xgb = Pipeline([('sc', StandardScaler()),
                    ('clf', XGBClassifier(scale_pos_weight=scale_pos, n_estimators=200,
                                         learning_rate=0.1, max_depth=5, random_state=42,
                                         eval_metric='logloss', verbosity=0))])
    results.append(evaluate('XGBoost', xgb, X_train, y_train, X_test, y_test))


In [3]:

# ── Comparison table ──
metrics_df = pd.DataFrame([{k:v for k,v in r.items() if k != 'pipeline'} for r in results])
print("=== Model Comparison Table ===")
print(metrics_df.to_string(index=False))

# ── Pick winner: highest recall, tiebreak on F1 ──
winner_idx = metrics_df['recall'].idxmax()
winner = results[winner_idx]
print(f"\n=== WINNER: {winner['model']} ===")
print(f"Reason: Highest Recall ({winner['recall']:.4f}) — missing a real flight-risk employee is the expensive mistake.")
print(f"Metrics: Precision={winner['precision']:.4f}, Recall={winner['recall']:.4f}, "
      f"F1={winner['f1']:.4f}, ROC-AUC={winner['roc_auc']:.4f}")


=== Model Comparison Table ===
              model  precision  recall     f1  roc_auc
Logistic Regression        0.7  0.6364 0.6667   0.9469
      Random Forest        1.0  0.6364 0.7778   1.0000
            XGBoost        1.0  1.0000 1.0000   1.0000

=== WINNER: XGBoost ===
Reason: Highest Recall (1.0000) — missing a real flight-risk employee is the expensive mistake.
Metrics: Precision=1.0000, Recall=1.0000, F1=1.0000, ROC-AUC=1.0000


In [4]:

# ── Save winning model ──
best_pipeline = winner['pipeline']
joblib.dump(best_pipeline, f'{MODELS}/attrition_pipeline.joblib')
print(f"Saved: {MODELS}/attrition_pipeline.joblib")

# Save comparison metrics
comparison = [{k:v for k,v in r.items() if k != 'pipeline'} for r in results]
with open(f'{MODELS}/model_comparison.json', 'w') as f:
    json.dump({'results': comparison, 'winner': winner['model'],
               'selection_criterion': 'highest recall (recall > f1 > precision)'}, f, indent=2)
print("Saved model_comparison.json")


Saved: ../models/attrition_pipeline.joblib
Saved model_comparison.json


**Model comparison complete.** Winner selected by highest recall. Pipeline saved to models/attrition_pipeline.joblib.